# rPPG deepfake detector — improved (v2.0)

This notebook **uses the same detector code as the web app** (`backend/app/detector`) instead of a copy.
The untouched original is in `notebooks/original/`. Nothing here measures accuracy; thresholds are uncalibrated.

Run it from the `notebooks/` folder with the backend virtual environment active (`pip install jupyter matplotlib` first).

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'backend'))
import numpy as np
import matplotlib.pyplot as plt
from app.detector import analyze_traces, analyze_video, DetectorConfig
print('Detector loaded')

## 1. Synthetic sanity check
Generated signals only verify that the code behaves as designed. They say nothing about real-world accuracy.

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(0, 30, 1/30)
def pulse(ph=0, f=1.2): return 128 + np.sin(2*np.pi*f*t+ph) + rng.normal(0, .35, len(t))
cases = {
  'consistent pulse': {'forehead': pulse(0), 'left_cheek': pulse(.1), 'right_cheek': pulse(.15)},
  'pure noise':       {k: 128 + rng.normal(0, 1, len(t)) for k in ('forehead','left_cheek','right_cheek')},
  'regions disagree': {'forehead': pulse(0), 'left_cheek': pulse(.1), 'right_cheek': pulse(0, f=2.1)},
}
for name, rois in cases.items():
    r = analyze_traces(t, rois, None, 0.05, 'mediapipe')
    print(f"{name:18s} -> {r['verdict']:13s} SNR={r['quality']['fused_snr_db']:.1f} dB")

## 2. Analyse a real video
Put a path below. Uses MediaPipe if `backend/models/face_landmarker.task` exists, otherwise Haar.

In [ ]:
VIDEO = 'your_video.mp4'   # <- change me
MODEL = Path('..').resolve() / 'backend' / 'models' / 'face_landmarker.task'
result, thumb = analyze_video(VIDEO, MODEL if MODEL.exists() else None)
print(result['headline'])
for x in result['reasons']: print(f" [{x['kind']}] {x['text']}")
print('Warnings:', result['warnings'])

In [ ]:
s, sp = result['signals'], result['spectrum']
if s:
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
    for k in ('forehead','left_cheek','right_cheek'): ax[0].plot(s['t'], s[k], lw=.8, label=k)
    ax[0].set_xlabel('s'); ax[0].legend(); ax[0].set_title('Filtered signals')
    ax[1].plot(sp['freqs'], sp['fused']); ax[1].axvline(sp['peak_hz'], ls='--', c='r'); ax[1].set_title('Spectrum'); ax[1].set_xlabel('Hz')
    plt.show()
else:
    print('No signal could be extracted from this video.')